# Customer Churn Prediction Using Machine Learning

**IBM SkillsBuild Data Analytics with AI Internship 2026**  
**Student:** Anubhab Nandi  
**Institution:** Guru Nanak Institute of Technology  

### Project Objective
The objective of this project is to analyze customer data, identify patterns related to customer churn, and build a machine learning model that predicts whether a customer is likely to leave a service.

### Technologies Used
- Python
- Pandas
- NumPy
- Matplotlib
- Scikit-learn
- Jupyter Notebook

### Machine Learning Algorithm
Random Forest Classifier


In [ ]:
# Install required libraries if needed
# Run this cell only if your environment does not already have the libraries.

%pip install pandas numpy matplotlib scikit-learn -q


In [ ]:
# Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Libraries imported successfully.")


In [ ]:
# Download and load the public Telco Customer Churn dataset

dataset_url = "https://raw.githubusercontent.com/IBM/employee-attrition-aif360/master/data/WA_Fn-UseC_-Telco-Customer-Churn.csv"

df = pd.read_csv(dataset_url)

print("Dataset loaded successfully.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

df.head()


In [ ]:
# Basic information about the dataset

print("Dataset Shape:", df.shape)
print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())


In [ ]:
# Data cleaning

# Convert TotalCharges from text to numeric.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Fill missing TotalCharges using the median.
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].median())

# Convert SeniorCitizen from 0/1 to readable labels.
df["SeniorCitizen"] = df["SeniorCitizen"].map({0: "No", 1: "Yes"})

# Remove customer ID because it does not help prediction.
df = df.drop(columns=["customerID"])

print("Data cleaning completed.")
print("Remaining missing values:", df.isnull().sum().sum())


In [ ]:
# Descriptive statistics

df.describe(include="all").T


In [ ]:
# Churn distribution

churn_counts = df["Churn"].value_counts()

print(churn_counts)

plt.figure(figsize=(7, 5))
plt.bar(churn_counts.index, churn_counts.values)
plt.title("Customer Churn Distribution")
plt.xlabel("Churn")
plt.ylabel("Number of Customers")
plt.show()


In [ ]:
# Churn percentage

churn_percentage = df["Churn"].value_counts(normalize=True) * 100

print("Customer Churn Percentage:")
for category, percentage in churn_percentage.items():
    print(f"{category}: {percentage:.2f}%")


In [ ]:
# Monthly charges by churn status

plt.figure(figsize=(8, 5))

for status in ["No", "Yes"]:
    subset = df[df["Churn"] == status]["MonthlyCharges"]
    plt.hist(subset, bins=25, alpha=0.6, label=f"Churn = {status}")

plt.title("Monthly Charges and Customer Churn")
plt.xlabel("Monthly Charges")
plt.ylabel("Number of Customers")
plt.legend()
plt.show()


In [ ]:
# Contract type and churn

contract_churn = pd.crosstab(df["Contract"], df["Churn"])

print(contract_churn)

contract_churn.plot(kind="bar", figsize=(9, 5))
plt.title("Customer Churn by Contract Type")
plt.xlabel("Contract Type")
plt.ylabel("Number of Customers")
plt.xticks(rotation=0)
plt.legend(title="Churn")
plt.tight_layout()
plt.show()


In [ ]:
# Encode categorical variables

# Convert all categorical columns into numerical dummy variables.
model_df = pd.get_dummies(df, drop_first=True)

# Convert boolean columns to integers.
model_df = model_df.astype(int)

print("Encoded dataset shape:", model_df.shape)
model_df.head()


In [ ]:
# Separate features and target

target_column = "Churn_Yes"

X = model_df.drop(columns=[target_column])
y = model_df[target_column]

print("Number of features:", X.shape[1])
print("Target distribution:")
print(y.value_counts())


In [ ]:
# Split data into training and testing sets

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))


In [ ]:
# Feature scaling

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature scaling completed.")


In [ ]:
# Train Random Forest model

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train_scaled, y_train)

print("Random Forest model trained successfully.")


In [ ]:
# Make predictions

y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)

print(f"Model Accuracy: {accuracy * 100:.2f}%")


In [ ]:
# Detailed model evaluation

print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Stayed", "Churned"]
))


In [ ]:
# Confusion matrix

cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Stayed", "Churned"]
)

disp.plot()
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# Feature importance

feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print("Top 15 important features:")
display(feature_importance.head(15))


In [ ]:
# Visualize top 10 features

top_features = feature_importance.head(10).sort_values("Importance")

plt.figure(figsize=(9, 6))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.title("Top 10 Features Influencing Churn Prediction")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Example prediction for one customer from the test set

sample = X_test.iloc[[0]]
sample_scaled = scaler.transform(sample)

prediction = model.predict(sample_scaled)[0]
probability = model.predict_proba(sample_scaled)[0][1]

if prediction == 1:
    result = "Customer is predicted to CHURN"
else:
    result = "Customer is predicted to STAY"

print(result)
print(f"Estimated churn probability: {probability * 100:.2f}%")


## Key Findings

The analysis shows that customer churn can be studied using customer demographics, service subscriptions, contract information, tenure, and billing-related features.

The Random Forest model learns patterns from the historical customer data and predicts the churn class for unseen customers. Feature importance helps identify which variables contribute most to the model's predictions.

### Conclusion

Machine learning can support customer retention analysis by identifying customers who show patterns associated with churn. Businesses can use such predictions to investigate customer needs and design appropriate retention strategies.

**Note:** Model accuracy is dependent on the dataset and train/test split. The model output should be treated as analytical support rather than a guarantee of future customer behavior.
